In [0]:

# ============================================================
# RetailPulse - Day 2: Customers Dataset Profiling
# ============================================================

# 1. Import PySpark functions
import pyspark.sql.functions as F


# 2. Define the dataset path
common_path = "/Volumes/retailpulse/dev/olist_raw"

customers_path = common_path + "/olist_customers_dataset.csv"


# 3. Load the customers CSV into a Spark DataFrame
customers_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(customers_path)
)


# 4. Display sample records
display(customers_df)


# 5. Inspect the schema
customers_df.printSchema()


# 6. Count total rows
customer_row_count = customers_df.count()

print("Total customer records:", customer_row_count)


# 7. Count NULL values in every column
customer_nulls = [
    F.coalesce(
        F.sum(
            F.when(F.col(c).isNull(), 1)
        ),
        F.lit(0)
    ).alias(c)
    for c in customers_df.columns
]

display(customers_df.select(customer_nulls))


# 8. Find duplicate customer_id groups
duplicate_customer_df = (
    customers_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_customer_count = duplicate_customer_df.count()

print("Duplicate customer_id groups:", duplicate_customer_count)


# 9. Find repeated customer_unique_id groups
duplicate_customerunq_df = (
    customers_df
    .groupBy("customer_unique_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_unique_id_count = duplicate_customerunq_df.count()

print("Repeated customer_unique_id groups:", duplicate_unique_id_count)


# 10. Count rows belonging to repeated customer_unique_id groups
repeated_customer_rows = (
    duplicate_customerunq_df
    .agg(F.sum("count").alias("total_rows"))
)

display(repeated_customer_rows)


# 11. Find the maximum occurrences of a customer_unique_id
max_unique_id_count = (
    duplicate_customerunq_df
    .agg(F.max("count").alias("max_occurrences"))
)

display(max_unique_id_count)


# 12. Check Distinct Records
custome_distinct_row_count=customers_df.distinct().count()
display("distinct_records_count : ", custome_distinct_row_count)

# 13. Duplicate Records Count
display("Duplicate Records Count : ",customer_row_count - custome_distinct_row_count)




# 1. Customers Dataset Profiling

## Objective
Analyze the Olist customers dataset to understand its structure, completeness, and uniqueness before processing it in the RetailPulse pipeline.

## Dataset Details
- **Source file:** `olist_customers_dataset.csv`
- **DataFrame:** `customers_df`
- **Total records:** 99,441

## Profiling Checks
1. Inspect sample records and schema.
2. Count total rows.
3. Calculate NULL counts for all columns.
4. Check duplicate `customer_id` groups.
5. Analyze repeated `customer_unique_id` values.
6. Check exact duplicate records.

## Key Findings
- Duplicate `customer_id` groups: 0
- Repeated `customer_unique_id` groups: 2,997
- Rows belonging to repeated-ID groups: 6,342
- Maximum occurrences of one `customer_unique_id`: 17
- Exact duplicate records: 0

## Data Quality Notes
Repeated `customer_unique_id` values are not automatically errors. They can represent multiple records associated with the same underlying customer. Review the NULL-count output before finalizing the data-quality assessment.

In [0]:

# ============================================================
# RetailPulse - Day 2: Orders Dataset Profiling
# ============================================================


# 1. Load the orders dataset
orders_path = common_path + "/olist_orders_dataset.csv"

orders_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(orders_path)
)


# 2. Display sample records
display(orders_df.limit(5))


# 3. Inspect schema
orders_df.printSchema()


# 4. Count total records
orders_row_count = orders_df.count()

print("Total Orders Records:", orders_row_count)


# 5. Count NULL values in every column
orders_nulls = [
    F.coalesce(
        F.sum(F.when(F.col(c).isNull(), 1)),
        F.lit(0)
    ).alias(c)
    for c in orders_df.columns
]

display(orders_df.select(orders_nulls))


# 6. Check duplicate order_id groups
duplicate_orderid_df = (
    orders_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_orderid_count = duplicate_orderid_df.count()

print("Duplicate order_id groups:", duplicate_orderid_count)


# 7. Check exact duplicate records
distinct_orders_count = orders_df.distinct().count()

duplicate_records_count = orders_row_count - distinct_orders_count

print("Distinct Orders Records:", distinct_orders_count)
print("Exact Duplicate Records:", duplicate_records_count)


# 8. Count orders by status
order_status_count_df = (
    orders_df
    .groupBy("order_status")
    .count()
    .orderBy(F.col("count").desc())
)

display(order_status_count_df)


# 9. Count missing customer-delivery timestamps by status
order_delivery_null_counts_df = (
    orders_df
    .groupBy("order_status")
    .agg(
        F.coalesce(
            F.sum(
                F.when(
                    F.col("order_delivered_customer_date").isNull(), 1
                )
            ),
            F.lit(0)
        ).alias("null_count")
    )
    .orderBy(F.col("null_count").desc())
)

display(order_delivery_null_counts_df)


# 10. Inspect delivered orders missing customer-delivery timestamps
delivered_missing_delivery_df = orders_df.filter(
    (F.col("order_status") == "delivered") &
    (F.col("order_delivered_customer_date").isNull())
)

display(delivered_missing_delivery_df)


# 11. Count missing approval timestamps by status
order_approval_nulls_by_status_df = (
    orders_df
    .groupBy("order_status")
    .agg(
        F.coalesce(
            F.sum(
                F.when(F.col("order_approved_at").isNull(), 1)
            ),
            F.lit(0)
        ).alias("null_count")
    )
    .orderBy(F.col("null_count").desc())
)

display(order_approval_nulls_by_status_df)


# 12. Inspect delivered orders missing approval timestamps
delivered_missing_approval_df = orders_df.filter(
    (F.col("order_status") == "delivered") &
    (F.col("order_approved_at").isNull())
)

display(delivered_missing_approval_df)


# 13. Count delivered orders later than their estimated delivery timestamp
late_delivered_orders_df = orders_df.filter(
    (F.col("order_status") == "delivered") &
    (
        F.col("order_delivered_customer_date") >
        F.col("order_estimated_delivery_date")
    )
)

late_delivered_orders_count = late_delivered_orders_df.count()

print("Orders delivered after estimated delivery timestamp:",
      late_delivered_orders_count)


# 2. Orders Dataset Profiling

## Objective
Profile the Olist orders dataset to assess data completeness, order identifier uniqueness, order status distribution, and delivery timestamp consistency.

## Dataset
- **Source file:** `olist_orders_dataset.csv`
- **Source location:** `/Volumes/retailpulse/dev/olist_raw/`
- **DataFrame:** `orders_df`

## Profiling Checks
1. Load the CSV and inspect sample records.
2. Examine column names and data types.
3. Calculate the total number of records.
4. Count NULL values in every column.
5. Check duplicate `order_id` groups.
6. Identify exact duplicate records across all columns.
7. Summarize orders by `order_status`.
8. Investigate missing customer-delivery timestamps by status.
9. Investigate missing approval timestamps by status.
10. Identify delivered orders with missing delivery or approval timestamps.
11. Count delivered orders whose actual delivery timestamp is later than their estimated delivery timestamp.

## Key Findings
- Total records: 99,441
- Duplicate `order_id` groups: 0
- Exact duplicate records: 0
- Missing `order_approved_at`: 160
- Missing `order_delivered_carrier_date`: 1,783
- Missing `order_delivered_customer_date`: 2,965
- Delivered orders missing customer-delivery timestamps: 8
- Delivered orders missing approval timestamps: 14
- Delivered orders later than their estimated delivery timestamps: 7,826

## Data Quality Notes
Missing timestamps must be interpreted in the context of order status. The eight delivered orders missing customer-delivery timestamps and the fourteen delivered orders missing approval timestamps require investigation. Do not automatically fill in missing timestamps or remove records.

The late-delivery metric compares full timestamps, not just calendar dates.

## Outcome
The orders dataset has been profiled for basic completeness, uniqueness, status distribution, and delivery-related inconsistencies. Findings will inform validation rules in the RetailPulse Silver layer.